# Section 5.2 - Out-of-Sample Performance and Stability

This notebook generates the paper-ready figures for the out-of-sample comparison section.

Main question:
- which portfolio construction method performs better out-of-sample?

More important question:
- is that performance consistent or fragile?

Primary source:
- `backtest_summary.csv`

This section is designed to answer, explicitly or implicitly:
1. Which method has the best average performance?
2. Which method is most consistent across runs?
3. Does any method dominate?
4. How does Markowitz behave out of sample?
5. Is naive diversification competitive?


In [11]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

pio.renderers.default = "plotly_mimetype"

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "portafolios").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "portafolios").exists():
    raise RuntimeError("Could not locate the repository root containing the portafolios package.")

OUTPUT_CANDIDATES = [
    PROJECT_ROOT / "outputs" / "data_exports" / "final_experimental_setup",
    PROJECT_ROOT / "outputs" / "final_experimental_setup",
]
OUTPUT_DIR = next((path for path in OUTPUT_CANDIDATES if path.exists()), None)
if OUTPUT_DIR is None:
    raise RuntimeError("Could not locate final experimental setup outputs.")

TABLE_DIR = OUTPUT_DIR / "tables"
RUNS_DIR = OUTPUT_DIR / "framework_runs"
FIGURE_DIR = OUTPUT_DIR / "paper_figures" / "section_5_2"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

backtest = pd.read_csv(TABLE_DIR / "backtest_summary.csv")

METHOD_ORDER = [
    "equal_weight",
    "naive_risk_parity",
    "markowitz",
    "hrp_recursive",
    "hca_deprado_ew_nrp",
]

METHOD_LABELS = {
    "equal_weight": "Equal Weight (1/N)",
    "naive_risk_parity": "Naive Risk Parity",
    "markowitz": "Markowitz",
    "hrp_recursive": "HRP Recursive",
    "hca_deprado_ew_nrp": "HCA De Prado EW/NRP",
}

UNIVERSE_LABELS = {
    "djia": "DJIA",
    "sp100_style_top100": "S&P 100-style Top 100",
    "multi_asset_etf": "Multi-Asset ETF",
}

PALETTE = {
    "equal_weight": "#4C78A8",
    "naive_risk_parity": "#72B7B2",
    "markowitz": "#F58518",
    "hrp_recursive": "#54A24B",
    "hca_deprado_ew_nrp": "#E45756",
}


def method_label(method: str) -> str:
    return METHOD_LABELS.get(method, method)


def universe_label(universe_id: str) -> str:
    return UNIVERSE_LABELS.get(universe_id, universe_id)


def apply_paper_layout(fig: go.Figure, *, title: str, width: int = 1050, height: int = 620) -> go.Figure:
    fig.update_layout(
        title=title,
        template="plotly_white",
        width=width,
        height=height,
        font=dict(size=13),
        paper_bgcolor="white",
        plot_bgcolor="white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(l=70, r=40, t=90, b=65),
    )
    return fig


def save_figure(fig: go.Figure, filename: str) -> Path:
    output_path = FIGURE_DIR / filename
    fig.write_html(output_path, include_plotlyjs=True, full_html=True)
    display(fig)
    return output_path


bt = backtest.copy()
bt["method_label"] = bt["method"].map(method_label)
bt["universe_label"] = bt["universe_id"].map(universe_label)
bt["run_id"] = (
    bt["universe_id"].astype(str)
    + "_"
    + pd.to_datetime(bt["construction_date"]).dt.strftime("%Y%m%d")
    + "_"
    + bt["estimation_months"].astype(str)
    + "m"
)

print(f"Using output directory: {OUTPUT_DIR}")
print(f"Backtest rows: {len(bt)}")
print(f"Unique runs: {bt['run_id'].nunique()}")
print(f"Paper figures directory: {FIGURE_DIR}")

Using output directory: C:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup
Backtest rows: 135
Unique runs: 27
Paper figures directory: C:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup\paper_figures\section_5_2


## Summary Statistics For The Discussion

This table supports the written flow of the section:
- average performance,
- variability and stability,
- trade-offs,
- method-level interpretation.


In [13]:
summary_table = (
    bt.groupby(["method", "method_label"], as_index=False)
    .agg(
        mean_return=("annualized_return", "mean"),
        std_return=("annualized_return", "std"),
        mean_volatility=("volatility", "mean"),
        mean_sharpe=("sharpe_ratio", "mean"),
        std_sharpe=("sharpe_ratio", "std"),
        median_sharpe=("sharpe_ratio", "median"),
        mean_drawdown=("max_drawdown", "mean"),
        n_runs=("run_id", "nunique"),
    )
)
summary_table["sharpe_cv"] = summary_table["std_sharpe"] / summary_table["mean_sharpe"].replace(0, np.nan)
summary_table = summary_table.sort_values("mean_sharpe", ascending=False)
summary_table

,method,method_label,mean_return,std_return,mean_volatility,mean_sharpe,std_sharpe,median_sharpe,mean_drawdown,n_runs,sharpe_cv
0,equal_weight,Equal Weight (1/N),0.1376,0.2131,0.2073,0.8327,1.3717,0.5696,-0.1908,27,1.6473
4,naive_risk_parity,Naive Risk Parity,0.1155,0.1863,0.1878,0.7828,1.3132,0.4599,-0.1769,27,1.6775
1,hca_deprado_ew_nrp,HCA De Prado EW/NRP,0.0991,0.1860,0.1890,0.6170,1.1830,0.4781,-0.1836,27,1.9173
2,hrp_recursive,HRP Recursive,0.0653,0.1760,0.1910,0.5497,1.1596,0.3596,-0.1887,27,2.1096
3,markowitz,Markowitz,0.0911,0.3484,0.2389,0.4238,1.2425,0.3568,-0.2205,27,2.9316


## Figure 1 - Average Sharpe By Method

This is the primary answer to: who wins on average?

In [14]:
fig1_data = (
    bt.groupby(["method", "method_label"], as_index=False)["sharpe_ratio"]
    .mean()
    .sort_values("sharpe_ratio", ascending=False)
)

fig_avg_sharpe = go.Figure(
    data=[
        go.Bar(
            x=fig1_data["method_label"],
            y=fig1_data["sharpe_ratio"],
            marker=dict(color=[PALETTE[m] for m in fig1_data["method"]]),
            text=[f"{v:.2f}" for v in fig1_data["sharpe_ratio"]],
            textposition="outside",
        )
    ]
)
fig_avg_sharpe.update_xaxes(title="Method")
fig_avg_sharpe.update_yaxes(title="Average Sharpe ratio")
apply_paper_layout(fig_avg_sharpe, title="Average Out-of-Sample Sharpe Ratio by Method", height=560)
save_figure(fig_avg_sharpe, "figure_01_average_sharpe_by_method.html")

WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/data_exports/final_experimental_setup/paper_figures/section_5_2/figure_01_average_sharpe_by_method.html')

## Figure 2 - Sharpe Distribution By Method

This answers the stability question: who is consistent and who is fragile?

In [15]:
fig_box = px.box(
    bt,
    x="method_label",
    y="sharpe_ratio",
    color="method_label",
    points="all",
    color_discrete_map={method_label(k): v for k, v in PALETTE.items()},
)
fig_box.update_traces(jitter=0.22, pointpos=0, marker=dict(size=6, opacity=0.55))
fig_box.update_xaxes(title="Method")
fig_box.update_yaxes(title="Sharpe ratio")
apply_paper_layout(fig_box, title="Distribution of Out-of-Sample Sharpe Ratios by Method", height=620)
save_figure(fig_box, "figure_02_sharpe_distribution_boxplot.html")

WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/data_exports/final_experimental_setup/paper_figures/section_5_2/figure_02_sharpe_distribution_boxplot.html')

## Figure 3 - Return Versus Volatility

This is the efficiency trade-off figure: does higher return come with clearly higher risk?

In [16]:


fig_scatter = px.scatter(
    bt,
    x="volatility",
    y="annualized_return",
    color="method_label",
    hover_data=["run_id", "construction_date", "estimation_months", "sharpe_ratio", "max_drawdown"],
    color_discrete_map={method_label(k): v for k, v in PALETTE.items()},
)
fig_scatter.update_traces(marker=dict(size=11, line=dict(width=0.8, color="rgba(0,0,0,0.35)"), opacity=0.82))
fig_scatter.update_xaxes(title="Volatility")
fig_scatter.update_yaxes(title="Annualized return")
apply_paper_layout(fig_scatter, title="Out-of-Sample Return and Volatility Across Runs", width=1100, height=680)
save_figure(fig_scatter, "figure_03_return_vs_volatility_scatter.html")

WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/data_exports/final_experimental_setup/paper_figures/section_5_2/figure_03_return_vs_volatility_scatter.html')

## Figure 4 - Representative Run Cumulative Wealth

This figure shows what the comparison looks like in a concrete run.

Selection rule used here:
- choose the run with the highest average Sharpe ratio across methods.



In [17]:
run_quality = (
    bt.groupby("run_id", as_index=False)["sharpe_ratio"]
    .mean()
    .sort_values("sharpe_ratio", ascending=False)
)
RUN_ID = run_quality.iloc[0]["run_id"]

print("Selected representative run:", RUN_ID)
display(run_quality.head(10))

Selected representative run: sp100_style_top100_20200601_24m


,run_id,sharpe_ratio
22,sp100_style_top100_20200601_24m,2.5558
23,sp100_style_top100_20200601_36m,2.5320
21,sp100_style_top100_20200601_12m,2.4609
3,djia_20200601_12m,2.1073
4,djia_20200601_24m,2.0708
5,djia_20200601_36m,2.0482
12,multi_asset_etf_20200601_12m,1.6053
14,multi_asset_etf_20200601_36m,1.5970
13,multi_asset_etf_20200601_24m,1.5376
9,multi_asset_etf_20190601_12m,1.1698


In [18]:
fig_cumulative = go.Figure()

for method in METHOD_ORDER:
    path = RUNS_DIR / RUN_ID / "backtests" / method / "cumulative_returns.csv"
    if not path.exists():
        continue
    series = pd.read_csv(path, index_col=0)
    series.index = pd.to_datetime(series.index)
    fig_cumulative.add_trace(
        go.Scatter(
            x=series.index,
            y=series.iloc[:, 0],
            mode="lines",
            name=method_label(method),
            line=dict(width=2.6, color=PALETTE.get(method, "#444444")),
        )
    )

fig_cumulative.update_xaxes(title="Date")
fig_cumulative.update_yaxes(title="Cumulative wealth (start = 1.0)")
apply_paper_layout(fig_cumulative, title=f"Representative Out-of-Sample Backtest - {RUN_ID}", width=1150, height=650)
save_figure(fig_cumulative, "figure_04_representative_run_cumulative_wealth.html")

WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/data_exports/final_experimental_setup/paper_figures/section_5_2/figure_04_representative_run_cumulative_wealth.html')

## Generated Files

In [19]:
generated = sorted(FIGURE_DIR.glob("*.html"))
pd.DataFrame({"file": [p.name for p in generated], "path": [str(p) for p in generated]})

,file,path
0,figure_01_average_sharpe_by_method.html,C:\Users\narro\Desktop\semestre\honores_actuar...
1,figure_02_sharpe_distribution_boxplot.html,C:\Users\narro\Desktop\semestre\honores_actuar...
2,figure_03_return_vs_volatility_scatter.html,C:\Users\narro\Desktop\semestre\honores_actuar...
3,figure_04_representative_run_cumulative_wealth...,C:\Users\narro\Desktop\semestre\honores_actuar...
